# 🔮 Year-by-Year Coastal Classification

Run the trained model on **each year's satellite data** (2020-2025) to generate unique classified maps.

## What This Notebook Does:
- ✅ Load trained Random Forest model
- ✅ Loop through years 2020-2025
- ✅ For each year:
  - Load year-specific satellite bands
  - Compute spectral indices (NDVI, NDWI, etc.)
  - Extract spatial features
  - Generate classification map
  - Export to `../FRONTEND/data/classified_YYYY.png`
  - Calculate area statistics

## Output:
- `../FRONTEND/data/classified_2020.png` through `classified_2025.png`
- `../results/year_stats.json` - Area statistics for all years

---

## 1. Load Model and Setup

In [1]:
import rasterio
import numpy as np
import joblib
import json
import glob
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from scipy.ndimage import generic_filter, uniform_filter
from pathlib import Path

# Load trained model
print("📂 Loading trained model...")
model = joblib.load("../models/coastal_classifier_model.pkl")
scaler = joblib.load("../models/feature_scaler.pkl")

with open("../models/model_metadata.json") as f:
    metadata = json.load(f)

print(f"✅ Model loaded: {metadata['model_type']}")
print(f"   Test Accuracy: {metadata['test_accuracy']*100:.2f}%")
print(f"   Expected features: {metadata['n_features']}")
print(f"   Classes: {metadata['classes']}")

# Class definitions
CLASS_INFO = {
    1: {"name": "Seagrass",  "color": "#1a936f"},  # green (match frontend)
    2: {"name": "Sand",      "color": "#ffe156"},  # yellow
    3: {"name": "Seaweed",   "color": "#8e44ad"},  # purple
    4: {"name": "Water",     "color": "#0077b6"},  # blue
    5: {"name": "Landmass",  "color": "#8b4513"},  # brown
}

# Output folder for frontend
frontend_data = Path("../FRONTEND/data")
frontend_data.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Output folder: {frontend_data.absolute()}")

📂 Loading trained model...
✅ Model loaded: RandomForestClassifier (RF + Spatial Features)
   Test Accuracy: 93.12%
   Expected features: 21
   Classes: [1, 2, 3, 4, 5]

📁 Output folder: c:\Users\HP LAPTOP 15s\matlabTiff\notebooks\..\FRONTEND\data


## 2. Define Processing Functions

In [2]:
def find_band_file(year_folder, band_name):
    """Find band file in year folder (handles different naming patterns)."""
    patterns = [
        f"*_{band_name}_(Raw).tiff",
        f"*_{band_name}.tiff",
        f"*_{band_name}*.tif"
    ]
    
    for pattern in patterns:
        matches = glob.glob(str(year_folder / pattern))
        if matches:
            return matches[0]
    return None

def majority_filter(values):
    """Return the most common value in a neighborhood window."""
    vals = values[values > 0]  # ignore nodata (0)
    if len(vals) == 0:
        return 0
    counts = np.bincount(vals.astype(int))
    return np.argmax(counts)

def compute_indices(B02, B03, B04, B08):
    """Compute spectral indices: NDVI, NDWI, BAI."""
    # NDVI: (NIR - Red) / (NIR + Red)
    NDVI = (B08 - B04) / (B08 + B04 + 1e-10)
    
    # NDWI: (Green - NIR) / (Green + NIR) - water index
    NDWI = (B03 - B08) / (B03 + B08 + 1e-10)
    
    # BAI: 1/((0.1-Red)^2 + (0.06-NIR)^2) - bareness index
    BAI = 1.0 / ((0.1 - B04)**2 + (0.06 - B08)**2 + 1e-10)
    
    return NDVI, NDWI, BAI

def process_year(year):
    """Process one year's satellite data and generate classified map."""
    print(f"\n{'='*60}")
    print(f"📅 Processing Year: {year}")
    print(f"{'='*60}")
    
    year_folder = Path(f"../data/years/{year}")
    
    if not year_folder.exists():
        print(f"   ⚠️  Folder not found: {year_folder}")
        return None
    
    # Find required bands
    print("   🔍 Looking for band files...")
    B02_file = find_band_file(year_folder, "B02")
    B03_file = find_band_file(year_folder, "B03")
    B04_file = find_band_file(year_folder, "B04")
    B08_file = find_band_file(year_folder, "B08")
    
    if not all([B02_file, B03_file, B04_file, B08_file]):
        print(f"   ❌ Missing required bands for {year}")
        print(f"      B02: {B02_file}")
        print(f"      B03: {B03_file}")
        print(f"      B04: {B04_file}")
        print(f"      B08: {B08_file}")
        return None
    
    print(f"   ✅ Found all required bands")
    
    # Load bands
    print("   📂 Loading bands...")
    with rasterio.open(B02_file) as src:
        B02 = src.read(1).astype(float)
        profile = src.profile.copy()
    
    B03 = rasterio.open(B03_file).read(1).astype(float)
    B04 = rasterio.open(B04_file).read(1).astype(float)
    B08 = rasterio.open(B08_file).read(1).astype(float)
    
    height, width = B02.shape
    print(f"   📐 Image size: {height} × {width} pixels")
    
    # Compute spectral indices
    print("   🧮 Computing spectral indices...")
    NDVI, NDWI, BAI = compute_indices(B02, B03, B04, B08)
    
    # Stack all 7 bands
    image_data = np.stack([B02, B03, B04, B08, NDVI, NDWI, BAI], axis=0)
    
    # Identify nodata pixels
    nodata_mask = (
        np.all(image_data == 0, axis=0) |
        np.any(np.isnan(image_data), axis=0)
    )
    print(f"   🔍 Nodata pixels: {np.sum(nodata_mask):,}")
    
    # Compute spatial features (21 total)
    print("   🔧 Computing spatial neighborhood features...")
    neigh_means = np.zeros_like(image_data)
    neigh_stds = np.zeros_like(image_data)
    
    for b in range(7):
        band = image_data[b].astype('float32')
        mean = uniform_filter(band, size=3)
        sqr_mean = uniform_filter(band**2, size=3)
        var = np.maximum(sqr_mean - mean**2, 0)
        neigh_means[b] = mean
        neigh_stds[b] = np.sqrt(var)
    
    # Stack to 21 features and reshape
    spatial_stack = np.concatenate([image_data, neigh_means, neigh_stds], axis=0)
    features = spatial_stack.reshape(21, -1).T
    features = np.nan_to_num(features, nan=0)
    
    print(f"   📊 Feature array shape: {features.shape}")
    
    # Scale and predict
    print("   🔮 Making predictions...")
    features_scaled = scaler.transform(features)
    prediction_flat = model.predict(features_scaled)
    
    # Zero out nodata pixels
    prediction_flat[nodata_mask.flatten()] = 0
    
    # Reshape to map
    prediction_map = prediction_flat.reshape(height, width)
    
    # Apply majority filter to clean noise
    print("   🧹 Applying majority filter...")
    prediction_clean = generic_filter(
        prediction_map.astype(float),
        majority_filter,
        size=3
    ).astype(np.uint8)
    
    # Calculate statistics
    valid_pixels = prediction_clean[prediction_clean > 0]
    print(f"\n   📈 Class Distribution:")
    stats = {}
    for cls_id in sorted(np.unique(valid_pixels)):
        count = np.sum(prediction_clean == cls_id)
        pct = (count / len(valid_pixels)) * 100
        name = CLASS_INFO.get(int(cls_id), {}).get('name', f'Class {cls_id}')
        print(f"      {name}: {count:,} pixels ({pct:.1f}%)")
        stats[name] = int(count)
    
    # Generate visualization
    print("\n   🎨 Generating classified map...")
    
    # Build colormap
    unique_classes = sorted([c for c in np.unique(prediction_clean) if c > 0])
    max_class = max(CLASS_INFO.keys())
    colors = ['#ffffff']  # nodata = white
    for i in range(1, max_class + 1):
        if i in CLASS_INFO:
            colors.append(CLASS_INFO[i]["color"])
        else:
            colors.append('#ffffff')
    
    cmap = ListedColormap(colors)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(12, 9))
    
    ax.imshow(
        prediction_clean,
        cmap=cmap,
        vmin=0,
        vmax=max_class,
        interpolation='nearest'
    )
    
    # Add legend
    legend_patches = []
    for cls_id in unique_classes:
        if cls_id in CLASS_INFO:
            info = CLASS_INFO[cls_id]
            pixel_count = np.sum(prediction_clean == cls_id)
            total_valid = np.sum(prediction_clean > 0)
            pct = (pixel_count / total_valid) * 100
            label = f"{info['name']} ({pct:.1f}%)"
            patch = mpatches.Patch(color=info['color'], label=label)
            legend_patches.append(patch)
    
    ax.legend(
        handles=legend_patches,
        loc='lower right',
        fontsize=9,
        framealpha=0.9,
        title="Coastal Classes"
    )
    
    ax.set_title(f"Coastal Classification Map — {year}\n(Random Forest Model)",
                 fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    
    # Save to frontend folder
    output_path = frontend_data / f'classified_{year}.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"   💾 Saved: {output_path.name}")
    print(f"   ✅ Year {year} complete!")
    
    return stats

## 3. Process All Years

In [3]:
# Process years 2020-2025
years = [2020, 2021, 2022, 2023, 2024, 2025]
all_year_stats = {}

print("\n🚀 Starting year-by-year classification...\n")

for year in years:
    try:
        stats = process_year(year)
        if stats:
            all_year_stats[str(year)] = stats
    except Exception as e:
        print(f"\n   ❌ Error processing {year}: {e}")
        import traceback
        traceback.print_exc()

print(f"\n\n{'='*60}")
print("✅ ALL YEARS PROCESSED!")
print(f"{'='*60}")
print(f"\n📊 Processed {len(all_year_stats)} years: {list(all_year_stats.keys())}")
print(f"📁 Classified maps saved to: {frontend_data.absolute()}")


🚀 Starting year-by-year classification...


📅 Processing Year: 2020
   🔍 Looking for band files...
   ✅ Found all required bands
   📂 Loading bands...
   📐 Image size: 460 × 475 pixels
   🧮 Computing spectral indices...
   🔍 Nodata pixels: 0
   🔧 Computing spatial neighborhood features...
   📊 Feature array shape: (218500, 21)
   🔮 Making predictions...
   🧹 Applying majority filter...

   📈 Class Distribution:
      Seagrass: 160,994 pixels (73.7%)
      Sand: 57,506 pixels (26.3%)

   🎨 Generating classified map...
   💾 Saved: classified_2020.png
   ✅ Year 2020 complete!

📅 Processing Year: 2021
   🔍 Looking for band files...
   ✅ Found all required bands
   📂 Loading bands...
   📐 Image size: 460 × 475 pixels
   🧮 Computing spectral indices...
   🔍 Nodata pixels: 0
   🔧 Computing spatial neighborhood features...
   📊 Feature array shape: (218500, 21)
   🔮 Making predictions...
   🧹 Applying majority filter...

   📈 Class Distribution:
      Seagrass: 109,888 pixels (50.3%)
      Sa

## 4. Save Statistics (Optional)

In [4]:
# Save year statistics to JSON
results_folder = Path("../results")
results_folder.mkdir(exist_ok=True)

with open(results_folder / 'year_stats.json', 'w') as f:
    json.dump(all_year_stats, f, indent=2)

print(f"💾 Statistics saved to: {results_folder.absolute()}/year_stats.json")

# Display summary
print("\n📋 Summary by Year:")
print("="*60)
for year, stats in all_year_stats.items():
    print(f"\n{year}:")
    for cls_name, count in stats.items():
        print(f"  {cls_name:<12}: {count:>10,} pixels")

💾 Statistics saved to: c:\Users\HP LAPTOP 15s\matlabTiff\notebooks\..\results/year_stats.json

📋 Summary by Year:

2020:
  Seagrass    :    160,994 pixels
  Sand        :     57,506 pixels

2021:
  Seagrass    :    109,888 pixels
  Sand        :     54,567 pixels
  Seaweed     :        826 pixels
  Water       :     53,219 pixels

2022:
  Seagrass    :     43,573 pixels
  Sand        :     49,172 pixels
  Seaweed     :     95,306 pixels
  Water       :     30,449 pixels

2023:
  Seagrass    :    164,587 pixels
  Sand        :     53,899 pixels
  Seaweed     :         14 pixels

2025:
  Seagrass    :     15,650 pixels
  Sand        :     49,197 pixels
  Seaweed     :     40,437 pixels
  Water       :    113,216 pixels


## ✅ Done!

**Next Steps:**
1. Refresh your browser (Ctrl+F5) to see the new classified maps
2. Each year should now show a **unique** classified map based on its own satellite data
3. Compare the classified maps across years to see coastal changes!

**Note:** The area statistics on the dashboard are still using mock data. To update those, you'll need to modify notebook 06 to use the real statistics from `year_stats.json`.